# A crypto session, end to end

This notebook runs the whole ob-analytics pipeline on the **bundled Bitstamp BTC/USD sample** — load → reconstruct the book → plot → flow-toxicity metrics — using only the public API. It executes in well under a minute and ships as a living integration test (run in CI via `nbmake`).

Everything here applies unchanged to your own captures; see the [quickstart](../quickstart.md) for using your own data or LOBSTER.

In [ ]:
from ob_analytics import Pipeline, sample_csv_path

# One call: load the sample, set order types, build depth + summary.
result = Pipeline().run(sample_csv_path())
result.events.shape, result.trades.shape, result.depth.shape

## What can this result plot?

`available_concepts` reports the faces a given result supports and at which resolution(s) — it varies by data format (hidden executions, for instance, are LOBSTER-only).

In [ ]:
from ob_analytics.visualization import available_concepts

available_concepts(result)

## One-line plotting

`result.plot(concept, level=None, **overrides)` wires up the right prepare function and context for you. `level` defaults to L2; *comparable* concepts (both L2 and L3) take an explicit level. Extra keyword arguments flow through to the underlying prepare function.

In [ ]:
# Resting liquidity through time. col_bias<1 brightens thin levels.
result.plot("depth_heatmap", col_bias=0.3)

In [ ]:
# The L3 trade tape: signed lollipops + each consumed maker order's
# resting span.
result.plot("trade_tape", "L3")

## Flow-toxicity analytics

Analytic series are computed post-pipeline from `result.trades` and rendered with the same `plot` dispatcher via the public `prepare` namespace.

In [ ]:
from ob_analytics import compute_vpin, compute_kyle_lambda
from ob_analytics.visualization import plot, prepare

vpin = compute_vpin(result.trades, bucket_volume=5.0)
plot("vpin", **prepare.vpin(vpin, threshold=0.7))

In [ ]:
kyle = compute_kyle_lambda(result.trades, window="5min")
plot("kyle_lambda", **prepare.kyle_lambda(kyle))

## Where to go next

* `ob-analytics process <data> --gallery` renders **every** face to a standalone HTML gallery from the shell.
* `02_l3_microstructure.ipynb` digs into the L2-vs-L3 (Market-By-Price vs Market-By-Order) faces.
* Pass `backend="plotly"` to any `plot` / `result.plot` call for an interactive figure.